In [27]:
import pandas as pd                              # For data manipulation and analysis
import numpy as np                              # For numerical operations, especially with arrays

import yfinance as yf                          # To download financial market data from Yahoo Finance
from sentence_transformers import SentenceTransformer             # For generating sentence embeddings
from sklearn.decomposition import PCA                               # For Principal Component Analysis (dimensionality reduction)
from sklearn.metrics import accuracy_score                                 # To calculate the accuracy of a classification model
from lightgbm import LGBMClassifier                                       # For LightGBM classification models

# The next code block loads the 1st dataset and applies simple feature engineering to add lagged features and rolling statistics to better capture meaningful signal patterns for downstream forecasting tasks !



The two datasets used are:

Dow Jones Industrial Average: daily adjusted closing prices for 30 large-cap U.S. companies in the DJIA. We will retrieve this dataset using the yfinance library.
Combined_News_DJIA headlines: daily top-25 financial news headlines, largely including companies listed in the DJIA.

In [28]:
ticker = "^DJI"                                                          # Define the ticker symbol for the Dow Jones Industrial Average
df_price = yf.download(ticker, start="2008-01-01", end="2016-12-31")      # Download historical stock price data for the specified ticker and date range

df_price = df_price[["Close"]]                                            # Select only the 'Close' price column from the downloaded data
df_price["return"] = df_price["Close"].pct_change()                       # Calculate the daily percentage return from the 'Close' price
df_price["target"] = (df_price["return"].shift(-1) > 0).astype(int)        # Create a binary target variable: 1 if the next day's return is positive, 0 otherwise if negative

df_price = df_price.dropna()                                              # Remove any rows with NaN values, which result from pct_change() and shift(-1) operations
df_price.head()                                                           # Display the first few rows of the DataFrame to inspect the data

# Adding lagged and rolling average features

for lag in [1, 2, 3, 5]:                                                  # Iterate through a list of lag values
    df_price[f"lag_{lag}"] = df_price["return"].shift(lag)                # Create lagged return features for each specified lag

df_price["roll_mean_5"] = df_price["return"].rolling(5).mean()            # Calculate the 5-day rolling mean of the returns
df_price["roll_std_5"] = df_price["return"].rolling(5).std()                # Calculate the 5-day rolling standard deviation of the returns

df_price = df_price.dropna()                                               # Remove any remaining rows with NaN values introduced by rolling calculations

/tmp/ipython-input-537/390197641.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_price = yf.download(ticker, start="2008-01-01", end="2016-12-31")      # Download historical stock price data for the specified ticker and date range
[*********************100%***********************]  1 of 1 completed


In [29]:
# Load the news dataset from the specified URL into a pandas DataFrame
df_news = pd.read_csv("https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Combined_News_DJIA.csv" )

# Remove any leading/trailing whitespace from column names
df_news.columns = df_news.columns.str.strip()

# Convert the 'Date' column to datetime objects, assuming day comes first
df_news["Date"] = pd.to_datetime(df_news["Date"], dayfirst=True)

# Identify columns that contain news headlines (starting with 'Top')
headline_cols = [c for c in df_news.columns if c.startswith("Top")]

# Fill any missing values in the headline columns with an empty string
df_news[headline_cols] = df_news[headline_cols].fillna("")

# Create a new column 'combined' by concatenating all headline columns for each row
df_news["Combined Headline"] = df_news[headline_cols].apply( lambda row: " ".join([str(x) for x in row if str(x).strip() != ""]), # Join non-empty headline strings with a space
    axis=1) # Apply the function row-wise


df_news["Combined Headline"].head() # Display the first few entries of the newly created 'combined' headlines column

,Combined Headline
0,"b""Georgia 'downs two Russian warplanes' as cou..."
1,b'Why wont America and Nato help us? If they w...
2,b'Remember that adorable 9-year-old who sang a...
3,b' U.S. refuses Israel weapons to attack Iran:...
4,b'All the experts admit that we should legalis...


In [30]:
import ast

def clean_bytes_text(x):
    """
    Convert byte-like strings (b'...') to normal text
    """
    if isinstance(x, str):
        try:
            evaluated_value = ast.literal_eval(x)
            if isinstance(evaluated_value, bytes):
                return evaluated_value.decode('utf-8')
            else:
                return str(evaluated_value)
        except (ValueError, SyntaxError):
            if x.startswith("b"):
                return x.strip("b'\"")
            return x
    return x

df_news["Combined Headline"] = df_news["Combined Headline"].apply(clean_bytes_text)

In [31]:
# This block defines a helper function clean_bytes_text to process byte-like strings that appear in the raw text data
# (e.g., b'some text'). It attempts to evaluate the string as a Python literal and decode it if it's a byte string.
# If not, it handles cases where strings simply start with 'b' and removes them.
# This ensures that the text data is in a clean, readable format for further processing.

import ast

def clean_bytes_text(x):
    """
    Convert byte-like strings (b'...') to normal text
    """
    if isinstance(x, str):
        try:
            evaluated_value = ast.literal_eval(x)
            if isinstance(evaluated_value, bytes):
                return evaluated_value.decode('utf-8')
            else:
                return str(evaluated_value)
        except (ValueError, SyntaxError):
            if x.startswith("b"):
                return x.strip("b'\"")
            return x
    return x

df_news["combined"] = df_news["Combined Headline"].apply(clean_bytes_text) # Apply the cleaning function to the 'Combined Headline' column

# Using a pretrained transformer model to generate embeddings, passing the concatenated news headlines in the "combined" column as inputs.

In [32]:
# Initialize the pre-trained SentenceTransformer model for generating embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate sentence embeddings for the combined news headlines
embeddings = model.encode(
    df_news["combined"].tolist(), # Convert the 'combined' headline column to a list of strings
    show_progress_bar=True # Display a progress bar during the embedding generation process
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

# To reduce the risk of overfitting and limit the dimensionality of the embedding space, we apply principal component analysis (PCA).

In [33]:
pca = PCA(n_components=21)                                  # Initialize PCA with 21 components to reduce dimensionality
embeddings_reduced = pca.fit_transform(embeddings)          # Fit PCA to the embeddings and transform them into a lower-dimensional space

emb_df = pd.DataFrame(embeddings_reduced,index=df_news["Date"], columns=[f"emb_{i}" for i in range(21)])
# Create a new DataFrame from the reduced embeddings, using the 'Date' from df_news as index and labeling columns 'emb_0' to 'emb_19'.

# Merging the time series features with the generated embedding dimensions into a single dataset.

In [34]:
emb_df = emb_df.copy()

# Reset the index of the embeddings DataFrame to turn the 'Date' index into a column
emb_df.reset_index(inplace=True)

# Rename the first column (which was the index) to 'Date' for consistent merging
emb_df.rename(columns={emb_df.columns[0]: "Date"}, inplace=True)

# Handle potential extra index columns that might arise from multiple resets
if 'level_0' in df_price.columns:
    df_price = df_price.drop(columns=['level_0'])
if 'index' in df_price.columns:
    df_price = df_price.drop(columns=['index'])

# Reset the index of the price DataFrame to make 'Date' a column
df_price_cleaned = df_price.reset_index()

# Handle potential MultiIndex columns that can occur after yfinance download and reset_index
if isinstance(df_price_cleaned.columns, pd.MultiIndex):
    new_cols = []
    for col in df_price_cleaned.columns:
        if col[0] == 'Date':
            new_cols.append('Date')
        elif isinstance(col, tuple) and col[1] != '':
            new_cols.append(col[1])
        else:
            new_cols.append(col[0])
    df_price_cleaned.columns = new_cols

# Drop 'Ticker' column if it exists, as it's not needed for merging
if 'Ticker' in df_price_cleaned.columns:
    df_price_cleaned = df_price_cleaned.drop(columns=['Ticker'])

df_price_cleaned['Date'] = pd.to_datetime(df_price_cleaned['Date']) # Ensure the 'Date' column in df_price_cleaned is in datetime format

df = pd.merge(df_price_cleaned, emb_df, on="Date", how="inner") # Merge the cleaned price data with the embeddings DataFrame on the 'Date' column

# Separating the column names into two groups: those related to traditional time series attributes and those corresponding to LLM embeddings.

In [35]:
emb_features = [col for col in df.columns if "emb_" in col]

ts_features = [col for col in df.columns if "lag_" in col or "roll_" in col]

# Now training 2 forecasting models.

In [36]:
# IMPORTANT: set 'Date' as the DataFrame index for proper time series splitting
df = df.set_index('Date')

train = df[df.index < "2014-01-01"]  # Splitting threshold
test  = df[df.index >= "2014-01-01"]

X_train_base = train[ts_features]
X_test_base  = test[ts_features]

X_train_full = train[ts_features + emb_features]
X_test_full  = test[ts_features + emb_features]

y_train = train["target"]
y_test  = test["target"]

In [37]:
# Baseline model: no LLM embeddings
model_base = LGBMClassifier(random_state=42)
model_base.fit(X_train_base, y_train)

pred_base = model_base.predict(X_test_base)
acc_base = accuracy_score(y_test, pred_base)

print("Baseline Accuracy:", acc_base)

[LightGBM] [Info] Number of positive: 734, number of negative: 625
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000294 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 1359, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.540103 -> initscore=0.160757
[LightGBM] [Info] Start training from score 0.160757
Baseline Accuracy: 0.5


# The Baseline Accuracy of 0.5 (or 50%) tells us that our model, without the use of LLM embeddings, is performing no better than simply guessing the outcome at random. In a binary classification problem (like predicting whether the next day's return is positive or not, which has two possible outcomes), a 50% accuracy score means the model is essentially flipping a coin to make its predictions.

# It's a crucial benchmark:

# If your model achieves significantly higher than 0.5 accuracy, it indicates that the features you are using (in this case, ts_features like lagged returns and rolling statistics) have some predictive power.
# If your model is around 0.5 accuracy, as it is here, it suggests that the current ts_features alone are not sufficient to reliably predict the direction of the stock market movement. The model isn't finding any meaningful patterns in these features to make better-than-random predictions.
# This is why we often try more complex features or models, and why the next step will likely involve incorporating the LLM embeddings to see if they can improve this baseline.

In [38]:
# Full model with LLM embeddings
model_full = LGBMClassifier(random_state=42)
model_full.fit(X_train_full, y_train)

pred_full = model_full.predict(X_test_full)
acc_full = accuracy_score(y_test, pred_full)

print("With Embeddings Accuracy:", acc_full)

[LightGBM] [Info] Number of positive: 734, number of negative: 625
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000539 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6885
[LightGBM] [Info] Number of data points in the train set: 1359, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.540103 -> initscore=0.160757
[LightGBM] [Info] Start training from score 0.160757
With Embeddings Accuracy: 0.5142857142857142


# The 'With Embeddings Accuracy' is 0.5142857142857142 (approximately 51.4%).

# This tells us that when we incorporate the LLM (Sentence Transformer) embeddings into our model, the accuracy slightly increases from the baseline of 50% to about 51.4%. While this is a small improvement, it suggests that the news sentiment (captured by the embeddings) provides some additional predictive power beyond just the time series features.

# However, the improvement is not substantial enough to make the model highly predictive on its own, as it's still very close to random guessing. This could indicate:

# The chosen model (LGBMClassifier) might not be fully leveraging the embeddings.
# The embeddings themselves, or the way they've been reduced via PCA, might not be capturing the most relevant information for stock price prediction.
# Predicting daily stock movements is inherently very challenging, and even a small edge can be significant in real-world trading.


This project demonstrates the challenges and potential benefits of using LLM embeddings for financial forecasting. Here's what it tells us:

Difficulty of Forecasting: The baseline model, using only traditional time series features (lagged returns and rolling statistics), achieved an accuracy of 0.5. This highlights that predicting daily stock market movements is inherently very difficult, with basic historical price patterns offering no better prediction than a coin flip.

Value of LLM Embeddings: When we incorporated features derived from news headlines using LLM embeddings (after dimensionality reduction with PCA), the model's accuracy slightly improved to approximately 0.514. This small but measurable increase suggests that information extracted from news sentiment does carry some additional predictive signal, beyond what can be captured by purely numerical price-based features.

Room for Improvement: While the embeddings provided a positive lift, the overall accuracy is still close to random chance. This indicates that:

The current approach might not be fully capturing the nuances of market-moving news.
More sophisticated models or NLP techniques might be needed.
Other external factors not included in this analysis likely play a significant role in market movements.
In essence, the project shows that unstructured text data, when appropriately processed, can contribute to financial prediction, offering a slight edge. However, it also underscores the complexity of the task and the need for more advanced methodologies to achieve substantially better predictive power.